# SurfaceTarget-CRC — phase 01 — patch 1 14

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PATCH 1.14
#
# COHORT-WIDE RNA-SEQ PROCESSING CONSISTENCY AUDIT
#
# Objectives:
# 1. Retrieve cohort-wide GDC processing metadata
# 2. Examine STAR workflow-version distributions
# 3. Verify file metadata integrity
# 4. Compare duplicate candidates with the wider cohort
# 5. Evaluate protein-coding gene detection at CPM >= 1
#
# No final file selection.
# No additional molecular data downloads.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import requests
import json
import time

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
RESULTS = ROOT / "results/tables"

RAW = ROOT / "data/raw/tcga_duplicate_qc"

RESULTS.mkdir(parents=True, exist_ok=True)

TUMOR_PATH = META / "tcga_candidate_tumor_rnaseq_files.csv"
NORMAL_PATH = META / "tcga_candidate_normal_rnaseq_files.csv"

DUPLICATE_PATH = (
    META / "tcga_duplicate_rnaseq_provenance_audit.csv"
)

QC_PATH = (
    RESULTS / "tcga_duplicate_rnaseq_extended_qc.csv"
)

GDC_API = "https://api.gdc.cancer.gov/files"

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.14 — COHORT-WIDE WORKFLOW AUDIT")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load inventories
# ------------------------------------------------------------

tumor = pd.read_csv(
    TUMOR_PATH,
    dtype=str,
    keep_default_na=False
)

normal = pd.read_csv(
    NORMAL_PATH,
    dtype=str,
    keep_default_na=False
)

normal["cohort_tier"] = "Normal control"

inventory = pd.concat(
    [tumor, normal],
    ignore_index=True
)

if inventory["file_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate file IDs detected across inventories."
    )

print("\nCandidate tumor files:", len(tumor))
print("Normal tissue files:", len(normal))

print("Total unique files:", len(inventory))

# ------------------------------------------------------------
# 3. Retrieve GDC metadata in batches
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RETRIEVING COHORT-WIDE PROCESSING METADATA")
print("-" * 65)

session = requests.Session()

session.headers.update({
    "Accept": "application/json",
    "User-Agent": "SurfaceTarget-CRC-Research/1.0"
})

fields = [
    "file_id",
    "file_name",
    "file_size",
    "md5sum",
    "analysis.workflow_type",
    "analysis.workflow_version",
    "created_datetime",
    "updated_datetime"
]

file_ids = sorted(
    inventory["file_id"].unique()
)

BATCH_SIZE = 75

all_records = []

for start in range(0, len(file_ids), BATCH_SIZE):

    batch = file_ids[
        start:start + BATCH_SIZE
    ]

    filters = {
        "op": "in",
        "content": {
            "field": "file_id",
            "value": batch
        }
    }

    params = {
        "filters": json.dumps(filters),
        "fields": ",".join(fields),
        "format": "JSON",
        "size": len(batch)
    }

    for attempt in range(1, 4):

        try:

            response = session.get(
                GDC_API,
                params=params,
                timeout=90
            )

            response.raise_for_status()

            hits = response.json()["data"]["hits"]

            returned_ids = {
                item["file_id"]
                for item in hits
            }

            if returned_ids != set(batch):
                raise RuntimeError(
                    "GDC response missing requested files."
                )

            all_records.extend(hits)

            print(
                f"Retrieved {len(all_records)} "
                f"of {len(file_ids)} files"
            )

            break

        except Exception as e:

            if attempt == 3:
                raise

            print(
                f"Retry {attempt}: {e}"
            )

            time.sleep(2 ** attempt)

print("\nMetadata retrieval: COMPLETE")

# ------------------------------------------------------------
# 4. Extract processing metadata
# ------------------------------------------------------------

records = []

for item in all_records:

    analysis = item.get("analysis") or {}

    records.append({

        "file_id": item["file_id"],

        "gdc_file_name": item.get("file_name"),

        "gdc_file_size": item.get("file_size"),

        "gdc_md5sum": item.get("md5sum"),

        "workflow_type": analysis.get("workflow_type"),

        "workflow_version": analysis.get("workflow_version"),

        "created_datetime": item.get(
            "created_datetime"
        ),

        "updated_datetime": item.get(
            "updated_datetime"
        )
    })

metadata = pd.DataFrame(records)

if metadata["file_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate file IDs in GDC response."
    )

# ------------------------------------------------------------
# 5. Merge with cohort inventory
# ------------------------------------------------------------

merged = inventory.merge(
    metadata,
    on="file_id",
    how="left",
    validate="one_to_one"
)

if len(merged) != len(inventory):
    raise RuntimeError(
        "File count changed during metadata merge."
    )

if merged["workflow_type"].isna().any():
    raise RuntimeError(
        "Missing workflow metadata detected."
    )

# ------------------------------------------------------------
# 6. Verify file metadata
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("FILE METADATA VALIDATION")
print("-" * 65)

size_match = (
    pd.to_numeric(merged["file_size_bytes"])
    ==
    pd.to_numeric(merged["gdc_file_size"])
)

md5_match = (
    merged["md5sum"].str.lower()
    ==
    merged["gdc_md5sum"].str.lower()
)

workflow_match = (
    merged["workflow_type"] == "STAR - Counts"
)

print(
    "\nFile sizes matching:",
    int(size_match.sum()),
    "/",
    len(merged)
)

print(
    "MD5 checksums matching:",
    int(md5_match.sum()),
    "/",
    len(merged)
)

print(
    "STAR workflow confirmed:",
    int(workflow_match.sum()),
    "/",
    len(merged)
)

if not (
    size_match.all()
    and md5_match.all()
    and workflow_match.all()
):
    raise RuntimeError(
        "File metadata validation failed."
    )

print("\nFile metadata validation: PASSED")

# ------------------------------------------------------------
# 7. Examine workflow-version distribution
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("WORKFLOW VERSION DISTRIBUTION")
print("-" * 65)

merged["version_short"] = (
    merged["workflow_version"]
    .fillna("UNKNOWN")
    .astype(str)
    .str[:12]
)

version_summary = (
    merged.groupby(
        ["workflow_version"],
        dropna=False
    )
    .agg(
        n_files=("file_id", "nunique")
    )
    .reset_index()
    .sort_values(
        "n_files",
        ascending=False
    )
)

print(
    "\nUnique workflow versions:",
    merged["workflow_version"].nunique()
)

print("\nOverall distribution:")

print(
    version_summary.to_string(index=False)
)

# ------------------------------------------------------------
# 8. Project and sample-type distributions
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("WORKFLOW DISTRIBUTION BY PROJECT")
print("-" * 65)

project_versions = (
    merged.groupby(
        [
            "project",
            "sample_type",
            "version_short"
        ],
        dropna=False
    )
    .agg(
        files=("file_id", "nunique")
    )
    .reset_index()
)

print(
    "\n",
    project_versions.to_string(index=False)
)

# ------------------------------------------------------------
# 9. Investigate six duplicate candidates
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("DUPLICATE FILE WORKFLOW COMPARISON")
print("-" * 65)

duplicate_audit = pd.read_csv(
    DUPLICATE_PATH,
    dtype=str,
    keep_default_na=False
)

duplicate_ids = set(
    duplicate_audit["file_id"]
)

duplicate_versions = merged[
    merged["file_id"].isin(duplicate_ids)
].copy()

qc = pd.read_csv(QC_PATH)

qc_unstranded = qc[
    qc["count_mode"] == "unstranded"
].copy()

qc_columns = [
    "file_id",
    "detected_genes",
    "detected_protein_coding",
    "assignment_percent",
    "no_feature_percent"
]

duplicate_versions = duplicate_versions.merge(
    qc_unstranded[qc_columns],
    on="file_id",
    how="left",
    validate="one_to_one"
)

display_columns = [
    "patient_id",
    "sample_id",
    "file_id",
    "version_short",
    "detected_protein_coding",
    "assignment_percent",
    "no_feature_percent"
]

print(
    "\n",
    duplicate_versions[
        display_columns
    ]
    .round(2)
    .sort_values(
        ["patient_id", "sample_id", "file_id"]
    )
    .to_string(index=False)
)

# ------------------------------------------------------------
# 10. Protein-coding gene detection at CPM >= 1
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("PROTEIN-CODING EXPRESSION DETECTION")
print("-" * 65)

expression_records = []

summary_rows = {
    "N_unmapped",
    "N_multimapping",
    "N_noFeature",
    "N_ambiguous"
}

for _, row in duplicate_versions.iterrows():

    file_id = row["file_id"]

    path = RAW / f"{file_id}.tsv"

    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(
        path,
        sep="\t",
        comment="#",
        usecols=[
            "gene_id",
            "gene_type",
            "unstranded"
        ]
    )

    df = df[
        ~df["gene_id"].isin(summary_rows)
    ].copy()

    df["unstranded"] = pd.to_numeric(
        df["unstranded"],
        errors="raise"
    )

    total_assigned = df["unstranded"].sum()

    if total_assigned <= 0:
        raise RuntimeError(
            f"Invalid library counts: {file_id}"
        )

    coding = df[
        df["gene_type"] == "protein_coding"
    ].copy()

    coding["cpm"] = (
        coding["unstranded"]
        / total_assigned
        * 1_000_000
    )

    detected_cpm1 = int(
        (coding["cpm"] >= 1).sum()
    )

    detected_cpm5 = int(
        (coding["cpm"] >= 5).sum()
    )

    expression_records.append({

        "patient_id": row["patient_id"],

        "sample_id": row["sample_id"],

        "file_id": file_id,

        "file_short": file_id[:8],

        "protein_coding_cpm_ge_1":
            detected_cpm1,

        "protein_coding_cpm_ge_5":
            detected_cpm5
    })

expression_qc = pd.DataFrame(
    expression_records
)

print(
    "\n",
    expression_qc[
        [
            "patient_id",
            "sample_id",
            "file_short",
            "protein_coding_cpm_ge_1",
            "protein_coding_cpm_ge_5"
        ]
    ]
    .sort_values(
        ["patient_id", "sample_id", "file_short"]
    )
    .to_string(index=False)
)

print("\nNOTE:")
print(
    "CPM thresholds are descriptive detection "
    "criteria, not automatic exclusion rules."
)

# ------------------------------------------------------------
# 11. Save results
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING WORKFLOW AUDIT")
print("-" * 65)

merged.to_csv(
    META / "tcga_cohort_workflow_metadata.csv",
    index=False
)

version_summary.to_csv(
    RESULTS / "tcga_workflow_version_summary.csv",
    index=False
)

project_versions.to_csv(
    RESULTS / "tcga_workflow_project_summary.csv",
    index=False
)

duplicate_versions.to_csv(
    RESULTS / "tcga_duplicate_workflow_comparison.csv",
    index=False
)

expression_qc.to_csv(
    RESULTS / "tcga_duplicate_cpm_detection_qc.csv",
    index=False
)

# ------------------------------------------------------------
# 12. Save provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.14",

    "retrieved_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "NCI GDC API",

    "files_audited": len(merged),

    "unique_workflow_versions": int(
        merged["workflow_version"].nunique()
    ),

    "file_integrity_metadata_consistent": True,

    "duplicate_files_assessed": len(expression_qc),

    "cpm_thresholds": [1, 5],

    "count_column": "unstranded",

    "final_file_selection_performed": False,

    "limitations": [
        "Workflow version identifiers do not establish quality rankings.",
        "CPM-based detection is descriptive.",
        "RNA-seq library characteristics may differ.",
        "Sequencing-level QC has not been independently assessed.",
        "No final RNA-seq file selection was performed."
    ]
}

with open(
    META / "tcga_workflow_audit_provenance.json",
    "w"
) as f:

    json.dump(
        provenance,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 13. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.14 SUMMARY")
print("=" * 65)

print("\nFiles audited:", len(merged))

print(
    "Workflow versions identified:",
    merged["workflow_version"].nunique()
)

print("\nAll file metadata checks passed.")

print("\nSaved outputs:")

print("1. tcga_cohort_workflow_metadata.csv")
print("2. tcga_workflow_version_summary.csv")
print("3. tcga_workflow_project_summary.csv")
print("4. tcga_duplicate_workflow_comparison.csv")
print("5. tcga_duplicate_cpm_detection_qc.csv")
print("6. tcga_workflow_audit_provenance.json")

print("\nNo molecular datasets downloaded.")
print("No final RNA-seq file selection performed.")

print("\nPATCH 1.14 COMPLETE")
print("=" * 65)